In [47]:
import os

# Need to load in Keras 2 in order to use DenseVariational class which is required for BNN in Keras example
os.environ["TF_USE_LEGACY_KERAS"] = "1"


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
import tensorflow as tf
import tf_keras
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_probability as tfp
from scipy.stats import Normal



# Features are normalized by default
diabetes_sklearn = load_diabetes(scaled=False)
diabetes_df = pd.DataFrame(data = diabetes_sklearn.data,
                           columns = diabetes_sklearn.feature_names)

diabetes_df['target'] = diabetes_sklearn.target
y_m = diabetes_df['target'].mean()
y_std = diabetes_df['target'].std()

feat_names = diabetes_sklearn.feature_names

# Normalize the features around its mean explicitly
diabetes_df[feat_names] = (diabetes_df[feat_names] - diabetes_df[feat_names].mean()) / diabetes_df[feat_names].std()
# Normalize the target around its mean
diabetes_df['target'] = (diabetes_df['target'] - y_m) / y_std

# diabetes_df[diabetes_sklearn.feature_names] = (diabetes_df[diabetes_sklearn.feature_names] - diabetes_df[diabetes_sklearn.feature_names].mean()) / diabetes_df[diabetes_sklearn.feature_names].std() 

In [ ]:
train_df, test_df = train_test_split(diabetes_df, test_size = 0.2, random_state = 11242026) # create train and test set, shuffled
hidden_units = [4]
learning_rate = 0.001

def tf_translate_pandas(df, batch_size=32):
    df = df.copy()
    # create a dictionary of 1D arrays, each array are columns of the dataframe
    labels = df.pop('target') # the  target column
    features = {name : df[name].values.astype("float32") for name in df.columns}
    ds = tf.data.Dataset.from_tensor_slices((features, labels.values.astype("float32"))) # TensorFlow dataset

    return ds.batch(batch_size) # return in batches (useful for high volume datasets)

train_df = tf_translate_pandas(train_df)
test_df = tf_translate_pandas(test_df)


def run_experiment(model, loss, train_dataset, test_dataset):

    model.compile(
        optimizer=keras.optimizers.RMSprop(learning_rate=learning_rate),
        loss=loss,
        metrics=[keras.metrics.RootMeanSquaredError()],
    )

    print("Start training the model...")
    model.fit(train_dataset, epochs=num_epochs, validation_data=test_dataset)
    print("Model training finished.")
    _, rmse = model.evaluate(train_dataset, verbose=0)
    print(f"Train RMSE: {round(rmse, 3)}")

    print("Evaluating model performance...")
    _, rmse = model.evaluate(test_dataset, verbose=0)
    print(f"Test RMSE: {round(rmse, 3)}")

feats = diabetes_sklearn.feature_names

def create_model_inputs():
    inputs = {}
    for feature_name in feats:
        inputs[feature_name] = layers.Input(
            name=feature_name, shape=(1, ), dtype = tf.float32
        )
    return inputs

def compute_predictions(model, iterations = 100):
    predicted = []
    for _ in range(iterations):
        predicted.append(model(examples, training=False).numpy())
    predicted = np.concatenate(predicted, axis = 1)

    prediction_mean = np.mean(predicted, axis =1).tolist()
    prediction_min = np.min(predicted, axis=1).tolist()
    prediction_max = np.max(predicted, axis=1).tolist()
    prediction_range = (np.max(predicted, axis=1) - np.min(predicted, axis =1)).tolist()

    for idx in range(sample):
        print(
            f"Predictions mean: {round(prediction_mean[idx], 2)}, "
            f"min: {round(prediction_min[idx], 2)}, "
            f"max: {round(prediction_max[idx], 2)}, "
            f"range: {round(prediction_range[idx], 2)} - "
            f"Actual: {targets[idx]}"
        )

def predictive_summary(model, dataset, iterations = 100):
    x, y = list(dataset.unbatch().batch(1000))[0] # feature weights and targets extracted from full set
    preds = np.stack([model(x, training=False).numpy().ravel() for _ in range(iterations)]) # make predictions from 100 draws
    y = y.numpy().ravel() # flatten our target
    mean_pred, sd_pred = preds.mean(axis=0), preds.std(axis=0)
    rmse = np.sqrt(np.mean((mean_pred - y)**2)) # calculate the rmse manually
    low, high = np.percentile(preds, [2.5,97.5], axis=0) # get the 90% interval of predictions
    coverage = np.mean((low <= y) & (high >= y)) # compare the 90% interval to the test targets
    return rmse, sd_pred.mean(), coverage

compute_predictions(bnn_model_small)



## Method: Variational Inference

Posterior distribution: $P(\theta | X) = \frac{P(X | \theta) P(\theta)}{P(X)}$
- $\theta$: model parameters
- $X$: observed data
- $P(\theta | X)$: likelihood
- $P(\theta)$: prior
- $P(X)$ evidence (normalizing constant and is often ignored). 

When hard to compute we approximate the posterior with Variational Inference:

- introduces distribution $\phi(z)$ with parameters $\phi$
- We optimize the parameters by maximizing the Evidence Lower Bound (ELBO).

$$
\rm{ELBO} = \mathbb{E}_{q_{\phi}(z)} [\log p(x | z)] - D_{KL}(q_{\phi} (z) \ \| \ p(z))
$$

We regularize with $D_{KL}$ for accuracy and closeness to the prior

- $q(\theta)$: The approximate posterior distribution over network weights
- $\mathbb{E}_{q(\theta)}[\log p(\text{data}|\theta)]$: The expected log-likelihood so the model fits the data
- $\text{KL}(q(\theta) | p(\theta))$: The Kullback-Leibler divergence regularizes $q(\theta)$ to stay close to $p(\theta)$.

GeeksforGeeks. “Variational Inference in Bayesian Neural Networks.” 12:27:00+00:00. https://www.geeksforgeeks.org/deep-learning/variational-inference-in-bayesian-neural-networks/.


### Implementation

1. We must choose our priors, $p(\theta)$, and variational family, $q(\theta)$.

In this example we chose $p(\theta) \sim N(0,1)$ and $q(\theta) \sim \mathcal{N}(w | \mu, LL^\top)$ with learnable mean and variance per weight. 
Where L is the lower triangle scale matrix from `MultivariateNormalTriL` and $LL^\top$ is the covariance matrix. Both are learned during training

2. Sample Weights: For each mini-batch, sample weights $\theta$ from q(\theta).

3. Compute Expected Log-Likelihood: $\mathbb{E}_{q(\theta)} \left[ \log p(\text{data} \mid \theta) \right] \approx \frac{1}{S} \sum_{s=1}^{S} \log p(\text{data} \mid \theta^{(s)})$

    S = number of samples,
    $\theta^{(s)}$ = s-th sample from $q(\theta)$.

4. Compute KL Divergence: $\mathrm{KL}\left(q(\theta)\,\|\,p(\theta)\right)$

For Gaussians, this has a closed-form expression.

5. Optimize ELBO: Use stochastic gradient descent (SGD/Adam) to maximize ELBO (or equivalently, minimize $\text{ELBO}$).

GeeksforGeeks. “Variational Inference in Bayesian Neural Networks.” 12:27:00+00:00. https://www.geeksforgeeks.org/deep-learning/variational-inference-in-bayesian-neural-networks/.


In [ ]:
# Define our prior (Normal)

def prior(kernel_size, bias_size, dtype=None):
    n = kernel_size + bias_size # total number of weights and biases
    prior_model = keras.Sequential(
        [
            tfp.layers.DistributionLambda(
                # Uses the lambda function to assign each weight an independent N(0,1)
                # no training parameters, it is fixed
                lambda t : tfp.distributions.MultivariateNormalDiag(
                    # loc = mu, scale_diag = sigma
                    loc = tf.zeros(n), scale_diag = tf.ones(n) 
                    )
                )
            ]
    )
    return prior_model

# Define our approx_posterior
# It is a Gaussian with weights for covariance and mean, both trainable

def approx_posterior(kernel_size, bias_size, dtype=None):
    n = kernel_size + bias_size # our number of weights
    # kernel_size and bias_size is automatically calculated by DenseVariational
    posterior_model = keras.Sequential(
        [
            tfp.layers.VariableLayer(
                tfp.layers.MultivariateNormalTriL.params_size(n),
                dtype=dtype
            ),
            tfp.layers.MultivariateNormalTriL(n)
        ]
    )

def create_bnn_model(train_size):
    inputs = create_model_inputs()
    features = keras.layers.concatenate(list(inputs.values()))
    features = layers.BatchNormalization()(features) # normalize the features, doesn't matter if the features have already been normalized or scaled

    # Create hidden layers with weight uncertainty using the DenseVariation layer
    for units in hidden_units:
        features = tfp.layers.DenseVariational(
            units = units,
            make_prior_fn = prior,
            make_posterior_fn = approx_posterior,
            kl_weight = 1 / train_size,
            activation = 'sigmoid',
        )(features)

    # The outputs here are deterministic
    outputs = layers.Dense(units = 1)(features)
    model = keras.Model(inputs = inputs, outputs = outputs)
    return model    


# Full set
num_epochs = 500
mse_loss = keras.losses.MeanSquaredError()
bnn_model_full = create_bnn_model(len(diabetes_df) * 0.8)
run_experiment(bnn_model_full, mse_loss, train_df, test_df)


## Joint Distribution for Normal-Normal


$$
\begin{aligned}
&p(\mu, \sigma | y_1, \ldots, y_n) \propto L(\mu, \sigma) \cdot p(\mu, \sigma)
\\ 
\rm{where} 
\\
&p(\mu, \sigma) = p(\sigma) \cdot p(\mu | \sigma)
\end{aligned}
$$


Here, our prior mean: $w \sim N(0,1)$ for every weight.

Here, Our likelihood: $y_i \sim \mathcal{N}(f(x_i; w), \tau^2)$

Note, $f(x_i ; w)$ is the network output given input $x_i$. Also, the log posterior will NOT be linear or shape into a Normal distribution and will not have a closed form

## Log Likelihood for Normal Distribution

Recall our likelihood is: $y_i \sim \mathcal{N}(f(x_i ; w), \tau^2)$.

This implicitly assumes that that our observed data is generated: $y = f(x_i; w) + \epsilon_i$ where $\epsilon_i \sim \mathcal{N}(0, \tau^2)$ and $f$ is our output function (the network)

(Aside)
Fun fact, this looks just like a time series model since the white noise in most models there are Gaussian distributed with mean 0. Ex: $Y_t = C + \phi Y_{t-1} + Z_t$ where $Z_t \sim \mathcal{N}(0, \sigma^2)$

Chandra, Rohitash, and Joshua Simmons. “Bayesian Neural Networks via MCMC: A Python-Based Tutorial.” IEEE Access 12 (2024): 70519–49. https://doi.org/10.1109/ACCESS.2024.3401234.


In [ ]:
# Define our log likelihood
def get_loglik(x, y, theta, tausq, model):
    # theta is our proposed weights
    # tausq is our proposed observed noise / variance
    preds = model.evaluate_proposal(x, y)
    n = len(y)
    log_likelihood = np.sum()
    return(log_likelihood) 

In [ ]:
# We need joint probability for estimating mu and sigma
# joint 

def joint_prob_dist(x, n_hidden = 4):
    d = x.shape[1]
    n_w = d * n_hidden + n_hidden + 1 # number of unknowns 

    def net(w): # our f(x_i ; w)
        d

    joint_model = tfp.distributions.JointDistributionSequential(
        [
            tfp.distributions.Independent(
                tfp.distributions.Normal(loc = tf.zeros(n), scale = tf.ones(n)),
                reinterpreted_batch_ndims = 1,
            ),
            tfp.distributions.HalfNormal(scale = 1), # our prior sigma, to match the output scale,
            lambda sigma, w : tfp.distributions.Independent(
                tfp.distributions.Normal(loc = net(w), scale = sigma[..., None])
            ),
        ], 
    )
    return joint_model


In [ ]:
# Draft a Hamiltonian MCMC
def target_log_prob_fn(){
    
}
step_size = 0.01
num_leapfrog_steps = 10

kernel = tfp.mcmc.HamiltonianMonteCarlo(
    target_log_prob_fn = target_log_prob_fn,
    step_size = step_size,
    num_leapfrog_steps = num_leapfrog_steps
)

<KerasTensor: shape=(None, 10) dtype=float32 (created by layer 'batch_normalization_3')>